# 03 · Reranker로 결정 추천 만들기

**목표**: 사례와 정책 후보를 rerank하여 `approve`, `manual-review`, `decline` 중 하나를 **추천**합니다. reranker는 새 판단 규칙을 학습하는 모델이 아니라, 입력 사례와 후보 정책의 관련도를 점수화하는 모델입니다.

이 실습은 실제 승인·반려를 실행하지 않습니다. 점수 차이가 작거나 근거가 약하면 보수적으로 `manual-review`를 반환하고 담당자 검토를 요구합니다. 실행 전 루트 `.env`에 `LLM_MODEL`과 배포된 `RERANK_MODEL`을 설정하세요. LiteLLM/Cohere 호환 endpoint를 쓸 때만 `RERANK_ENDPOINT`, `RERANK_API_KEY`를 추가합니다.

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">1. 정책 기반 추천 만들기 (UI)</h2>
<p style="margin: 7px 0;">고객 행동을 입력하고 <b>추천 생성</b>을 누르세요. 결과는 정책 후보의 순위, score, margin, 사람 검토 필요 여부를 함께 보여줍니다.</p>
<ol style="margin: 7px 0 0; padding-left: 1.3em;"><li>고객 행동을 입력합니다.</li><li>필요하면 ⚙️ 설정에서 정책 문서, 최소 margin, 최소 score를 조정합니다.</li><li>추천 결과의 <code>requires_human_review</code>가 true이면 자동 승인·반려 대신 담당자 검토로 보냅니다.</li></ol>
<p style="margin: 7px 0;">UI는 추천과 근거 조회만 수행합니다. 승인·반려·검토 큐 등록 같은 외부 업무 상태는 변경하지 않습니다.</p>
  </div>
</div>

In [2]:
from pathlib import Path
import sys

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "shared").exists())
for module_path in (project_root / "shared", project_root / "2_gen_ai_hub_llm"):
    if str(module_path) not in sys.path:
        sys.path.insert(0, str(module_path))

from lab_common.gradio_ui import build_decision_reranker_ui
from src.reranker.lab_core import (
    APPLICATION_DECISIONS, DEFAULT_MIN_MARGIN, DEFAULT_MIN_SCORE,
    DEFAULT_ORDER_POLICY_DOCUMENT, run_decision_recommendation,
)

candidate_options = [
    {"id": candidate.id, "label": candidate.label, "description": candidate.description}
    for candidate in APPLICATION_DECISIONS
]
decision_ui = build_decision_reranker_ui(
    run_decision_recommendation,
    default_policy_document=DEFAULT_ORDER_POLICY_DOCUMENT,
    candidates=candidate_options,
    default_min_margin=DEFAULT_MIN_MARGIN,
    default_min_score=DEFAULT_MIN_SCORE,
)
decision_ui.launch(inline=True, share=False, show_error=True, server_name="127.0.0.1", width="100%", height=900)

* Running on local URL:  http://127.0.0.1:7878
* To create a public link, set `share=True` in `launch()`.


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1a73e8; background: #e8f0fe; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1a73e8; letter-spacing: .02em;">✅  체크포인트</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">체크포인트</h2>
<ul style="margin: 7px 0 0; padding-left: 1.3em;"><li>추천 결과에 정책 후보 순위와 <code>requires_human_review</code>가 보인다.</li><li>점수나 margin이 약한 결과는 <code>manual-review</code>로 반환된다.</li><li>UI가 추천만 생성하며 실제 업무 상태를 변경하지 않는다는 것을 확인한다.</li></ul>
<p style="margin: 7px 0;">문제가 없으면 다음으로 진행하세요. 아래 코드는 같은 코어 API를 Python으로 살펴보는 선택 심화 자료입니다.</p>
  </div>
</div>

<hr style="border: 0; border-top: 2px solid #c7d2e1; margin: 32px 0 14px;">
<div style="margin: 0 0 18px; color: #4b5f78; font-size: .86em; font-weight: 700; letter-spacing: .04em;">🧩 아래부터 코드 베이스 실습</div>

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #5f6368; background: #f1f3f4; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #5f6368; letter-spacing: .02em;">🔎  선택 심화</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">선택: 테스트된 코어 API로 추천 근거 읽기</h2>
<p style="margin: 7px 0;">아래 셀은 UI와 같은 <code>run_decision_recommendation()</code>을 호출합니다. LLM 정책 요약, reranker 호출, score·margin 검증, 사람 검토 게이트는 코어에 있으며 노트북에서 다시 구현하지 않습니다.</p>
  </div>
</div>

In [ ]:
from pathlib import Path
import sys

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "shared").exists())
for module_path in (project_root / "shared", project_root / "2_gen_ai_hub_llm"):
    if str(module_path) not in sys.path:
        sys.path.insert(0, str(module_path))

from lab_common.config import load_settings
from src.reranker.lab_core import run_decision_recommendation

settings = load_settings()

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">1. 사례를 정책 후보와 비교하기</h2>
<p style="margin: 7px 0;"><code>ranked_candidates</code>는 후보별 관련도 순위이고, <code>margin</code>은 1위와 2위의 점수 차이입니다. 원격 semantic reranker의 최소 score·margin은 운영 전에 라벨된 사례로 평가·조정해야 합니다.</p>
  </div>
</div>

In [ ]:
case = "인증된 고객이 주문 간격 60초 이상을 지키며 1분에 1회 주문을 시도했다. 중복, 실패, 한도 초과는 없다."
recommendation = run_decision_recommendation(case, settings=settings)
recommendation

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">2. 반환된 근거와 검토 게이트 읽기</h2>
<p style="margin: 7px 0;">코어가 반환한 recommendation, score, margin, 사람 검토 필요 여부를 그대로 확인합니다. <code>requires_human_review=True</code>이면 이후 워크플로에서 자동 상태 변경을 막아야 합니다.</p>
  </div>
</div>

In [ ]:
print(f"추천: {recommendation['recommendation_label']} ({recommendation['recommendation']})")
print(f"사람 검토 필요: {recommendation['requires_human_review']}")
print(f"margin: {recommendation['margin']:.3f}")
for candidate in recommendation["ranked_candidates"]:
    print(f"{candidate['rank']}. {candidate['label']}: {candidate['score']:.3f}")
print(recommendation["action"])

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #5f6368; background: #f1f3f4; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #5f6368; letter-spacing: .02em;">🔎  선택 심화</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">3. 불확실한 사례는 자동 실행하지 않기</h2>
<p style="margin: 7px 0;">누락·불일치·추가 검토가 필요한 사례도 동일한 공개 API에 전달합니다. 결과가 <code>manual-review</code>인지와 사람 검토 게이트를 함께 확인하세요.</p>
  </div>
</div>

In [ ]:
uncertain_case = "필수 서류가 누락되어 신원 정보와 제출 정보가 불일치한다. 추가 검토가 필요하다."
review_recommendation = run_decision_recommendation(uncertain_case, settings=settings)
review_recommendation

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1a73e8; background: #e8f0fe; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1a73e8; letter-spacing: .02em;">✅  체크포인트</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">운영에 연결할 때</h2>
<ol style="margin: 7px 0 0; padding-left: 1.3em;"><li>예제 후보 문구와 정책 문서를 조직에서 승인한 최신 문구로 교체합니다.</li><li>과거의 라벨된 사례로 후보 설명, 최소 score, 최소 margin을 평가합니다.</li><li><code>requires_human_review=True</code>인 결과는 자동 승인·반려 대신 검토 큐로 보냅니다.</li><li>실제 상태 변경은 이 추천 모듈 밖에서 권한 검증과 감사 로그를 거쳐 수행합니다.</li></ol>
  </div>
</div>